# Home Assignment 2
### Name: Anushka Sawant
### Matriculation Number: 100006644

# Document Search System using TF-IDF & Cosine Similarity
Type a query like **"space adventure with robots"** and the system finds the most relevant documents from the dataset.

## Step 0: Importing Packages

In [1]:
import re
import nltk
import numpy as np

nltk.download("stopwords", quiet=True)
nltk.download("wordnet",   quiet=True)
nltk.download("omw-1.4",   quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics.pairwise import cosine_similarity


## Step 1: Dataset Creation

In [2]:
documents = [
    {"title": "Galactic Odyssey",     "desc": "A thrilling space adventure with robots and alien worlds."},
    {"title": "The Chef's Secret",    "desc": "Learn to cook healthy food and delicious vegetarian meals."},
    {"title": "Cyber City 2099",      "desc": "A dark cyberpunk tale of hackers and advanced artificial intelligence."},
    {"title": "Yoga for Beginners",   "desc": "Simple exercises and meditation techniques for a healthy lifestyle."},
    {"title": "Mars Colony",          "desc": "Humanity's first settlement on the red planet faces a robotic uprising."},
    {"title": "The Daily Diet",       "desc": "A guide to balanced eating, healthy food choices, and nutrition."},
    {"title": "Deep Ocean",           "desc": "An underwater adventure exploring unknown alien-like species."},
    {"title": "Code Breakers",        "desc": "The history of hackers and computer science during wartime."},
    {"title": "Robot Gladiators",     "desc": "Action-packed movie where giant robots fight for entertainment."},
    {"title": "Healthy Hearts",       "desc": "Cardio workouts and healthy food recipes for a long life."}
]

print(f"Dataset loaded: {len(documents)} documents\n")
for doc in documents:
    print(f"  - {doc['title']}: {doc['desc']}")

Dataset loaded: 10 documents

  - Galactic Odyssey: A thrilling space adventure with robots and alien worlds.
  - The Chef's Secret: Learn to cook healthy food and delicious vegetarian meals.
  - Cyber City 2099: A dark cyberpunk tale of hackers and advanced artificial intelligence.
  - Yoga for Beginners: Simple exercises and meditation techniques for a healthy lifestyle.
  - Mars Colony: Humanity's first settlement on the red planet faces a robotic uprising.
  - The Daily Diet: A guide to balanced eating, healthy food choices, and nutrition.
  - Deep Ocean: An underwater adventure exploring unknown alien-like species.
  - Code Breakers: The history of hackers and computer science during wartime.
  - Robot Gladiators: Action-packed movie where giant robots fight for entertainment.
  - Healthy Hearts: Cardio workouts and healthy food recipes for a long life.


## Step 2:  Text Preprocessing
We clean all text by:
- Converting to **lowercase**
- Removing **punctuation**
- Removing **stopwords** (e.g. "the", "a", "for")
- **Lemmatizing** words to their base form (e.g. "robots" is "robot", "exercises" is "exercise")

In [3]:
stop_words  = set(stopwords.words("english"))
lemmatizer  = WordNetLemmatizer()


def preprocess(text):
    text  = text.lower()                          # lowercase
    text  = re.sub(r"[^a-z\s]", "", text)        # remove punctuation
    words = text.split()                          # split into words
    words = [
        lemmatizer.lemmatize(w)
        for w in words
        if w not in stop_words                    # remove stopwords & lemmatize
    ]
    return " ".join(words)


cleaned_docs = [preprocess(doc["desc"]) for doc in documents]

print("Before and after preprocessing:\n")
for i, doc in enumerate(documents):
    print(f"  [{doc['title']}]")
    print(f"    Original : {doc['desc']}")
    print(f"    Cleaned  : {cleaned_docs[i]}\n")

Before and after preprocessing:

  [Galactic Odyssey]
    Original : A thrilling space adventure with robots and alien worlds.
    Cleaned  : thrilling space adventure robot alien world

  [The Chef's Secret]
    Original : Learn to cook healthy food and delicious vegetarian meals.
    Cleaned  : learn cook healthy food delicious vegetarian meal

  [Cyber City 2099]
    Original : A dark cyberpunk tale of hackers and advanced artificial intelligence.
    Cleaned  : dark cyberpunk tale hacker advanced artificial intelligence

  [Yoga for Beginners]
    Original : Simple exercises and meditation techniques for a healthy lifestyle.
    Cleaned  : simple exercise meditation technique healthy lifestyle

  [Mars Colony]
    Original : Humanity's first settlement on the red planet faces a robotic uprising.
    Cleaned  : humanity first settlement red planet face robotic uprising

  [The Daily Diet]
    Original : A guide to balanced eating, healthy food choices, and nutrition.
    Cleaned  : 

## Step 3: Convert Text to TF-IDF Vectors

In [5]:
vectorizer  = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(cleaned_docs)

print(f"TF-IDF matrix shape : {tfidf_matrix.shape}")
print(f"   {tfidf_matrix.shape[0]} documents  x  {tfidf_matrix.shape[1]} unique words")

TF-IDF matrix shape : (10, 57)
   10 documents  x  57 unique words


## Steps 4, 5 & 6 — Search Function
Cleans the query → converts to TF-IDF vector → compares against all documents → returns **top 3 results**.

In [6]:
def search(query, top_n=3):
    print(f"\n Query: '{query}'")
    print("-" * 55)

    # Step 4 — preprocess the query the same way as the documents
    cleaned_query = preprocess(query)
    print(f"   Cleaned query: '{cleaned_query}'\n")

    # Step 5 — turn query into a TF-IDF vector using the same fitted vectorizer
    query_vector  = vectorizer.transform([cleaned_query])

    # Step 5 — calculate cosine similarity against all document vectors
    similarities  = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # Step 6 — sort by highest score and pick top N
    top_indices   = np.argsort(similarities)[::-1][:top_n]

    for rank, idx in enumerate(top_indices, start=1):
        score = similarities[idx]
        doc   = documents[idx]
        print(f"  #{rank}  {doc['title']}")
        print(f"       {doc['desc']}")
        print(f"        Similarity Score: {score:.3f}\n")

    if similarities[top_indices[0]] == 0:
        print(" No relevant results found. Try different keywords.")

## Queries

In [7]:
search("space adventure with robots")


 Query: 'space adventure with robots'
-------------------------------------------------------
   Cleaned query: 'space adventure robot'

  #1  Galactic Odyssey
       A thrilling space adventure with robots and alien worlds.
        Similarity Score: 0.670

  #2  Robot Gladiators
       Action-packed movie where giant robots fight for entertainment.
        Similarity Score: 0.193

  #3  Deep Ocean
       An underwater adventure exploring unknown alien-like species.
        Similarity Score: 0.193



In [8]:
search("healthy food")


 Query: 'healthy food'
-------------------------------------------------------
   Cleaned query: 'healthy food'

  #1  Healthy Hearts
       Cardio workouts and healthy food recipes for a long life.
        Similarity Score: 0.407

  #2  The Daily Diet
       A guide to balanced eating, healthy food choices, and nutrition.
        Similarity Score: 0.407

  #3  The Chef's Secret
       Learn to cook healthy food and delicious vegetarian meals.
        Similarity Score: 0.407



In [9]:
search("hackers and artificial intelligence")


 Query: 'hackers and artificial intelligence'
-------------------------------------------------------
   Cleaned query: 'hacker artificial intelligence'

  #1  Cyber City 2099
       A dark cyberpunk tale of hackers and advanced artificial intelligence.
        Similarity Score: 0.636

  #2  Code Breakers
       The history of hackers and computer science during wartime.
        Similarity Score: 0.202

  #3  Robot Gladiators
       Action-packed movie where giant robots fight for entertainment.
        Similarity Score: 0.000



In [10]:
# Try your own query!
user_query = input("Enter your search query: ")
search(user_query)


 Query: 'robots and entertainment'
-------------------------------------------------------
   Cleaned query: 'robot entertainment'

  #1  Robot Gladiators
       Action-packed movie where giant robots fight for entertainment.
        Similarity Score: 0.549

  #2  Galactic Odyssey
       A thrilling space adventure with robots and alien worlds.
        Similarity Score: 0.236

  #3  Healthy Hearts
       Cardio workouts and healthy food recipes for a long life.
        Similarity Score: 0.000



## Interpretation
- The TF-IDF and cosine similarity search system worked well for matching keywords. When searching "space adventure with robots" it correctly returned Galactic Odyssey and Robot Gladiators as top results because they share the most relevant words with the query.
- The main limitation is that it only matches exact words, not meaning. Searching "fitness and exercise" would not rank Yoga for Beginners highly even though it is clearly about that topic, simply because the word exercise does not appear in its description. It also cannot connect "AI" with "artificial intelligence" since it treats them as completely different terms.
- This shows that TF-IDF is a solid starting point but it does not truly understand language, it just counts and compares words. For smarter results you would need something like BERT which actually understands that related words and phrases mean similar things. Overall this task gave a practical understanding of how basic search engines work and why modern ones have moved far beyond simple word matching.